# Ingestors Deep Dive

Ingestors are the first stage of the document-graph pipeline. They prepare raw DataFrames
for transformation by selecting, renaming, reordering, and filtering columns and rows.

This notebook tests all seven ingestor providers:

| Provider | Category | Purpose |
|----------|----------|--------|
| ColumnSelectorProvider | Column | Keep/drop specific columns |
| ColumnRenamerProvider | Column | Rename columns to match graph schema |
| ColumnReorderProvider | Column | Reorder columns for consistent processing |
| ColumnTypeConverterProvider | Column | Convert dtypes (string→int, string→datetime) |
| SkipRowProvider | Row | Filter rows by field conditions |
| DateRangeFilterProvider | Row | Filter rows by date range |
| NumericIdCleanupIngestor | Field | Remove .0 suffix from float-typed IDs |

**Prerequisites:**
- \ (base install)
- No Neptune or AWS credentials required


In [ ]:
import pandas as pd
from graphrag_toolkit.document_graph.ingest.ingestors_provider_config import IngestorProviderConfig
from graphrag_toolkit.document_graph.ingest.column.column_selector import ColumnSelectorProvider
from graphrag_toolkit.document_graph.ingest.column.column_renamer import ColumnRenamerProvider
from graphrag_toolkit.document_graph.ingest.column.column_reorder import ColumnReorderProvider
from graphrag_toolkit.document_graph.ingest.column.column_type_converter import ColumnTypeConverterProvider
from graphrag_toolkit.document_graph.ingest.row.skip_row import SkipRowProvider
from graphrag_toolkit.document_graph.ingest.row.date_range_filter import DateRangeFilterProvider
from graphrag_toolkit.document_graph.ingest.field.numeric_id_cleanup_ingestor import NumericIdCleanupIngestor

In [ ]:
# Sample security/compliance data
df = pd.DataFrame({
    'user_id': [1001.0, 2002.0, 3003.0, 4004.0],
    'username': ['admin_user', 'analyst_01', 'auditor_03', 'service_acct'],
    'role': ['Administrator', 'Security Analyst', 'Compliance Auditor', 'Service Account'],
    'access_level': ['5', '3', '4', '2'],
    'status': ['active', 'active', 'inactive', 'suspended'],
    'last_login': ['2026-06-20', '2026-06-22', '2026-01-15', '2025-12-01'],
    'department': ['IT Security', 'SOC', 'GRC', 'DevOps'],
    'notes': ['Primary admin', 'L2 analyst', 'Annual review pending', 'Disabled']
})
print("Source DataFrame:")
print(df.to_string())

## Column Selector

Keep only the columns that will become node properties or identifiers. Drop irrelevant columns early to reduce pipeline complexity.


In [ ]:
config = IngestorProviderConfig(name='select_columns', type='column', args={
    'action': 'select',
    'columns': ['user_id', 'username', 'role', 'status']
})
ingestor = ColumnSelectorProvider(config)
result = ingestor.ingest(df)
print("ColumnSelectorProvider (select):")
print(result.to_string())
print(f"\nColumns retained: {list(result.columns)}")

## Column Renamer

Rename source columns to match your target graph schema. For example, rename \ → \ when your graph model uses different terminology than the source data.


In [ ]:
config = IngestorProviderConfig(name='rename_columns', type='column', args={
    'mapping': {
        'user_id': 'account_id',
        'username': 'principal_name',
        'access_level': 'clearance_level'
    }
})
ingestor = ColumnRenamerProvider(config)
result = ingestor.ingest(df)
print("ColumnRenamerProvider:")
print(result.columns.tolist())

## Column Reorder

Reorder columns for consistent downstream processing. Some constructors assume a specific column order (e.g., first column = node ID).


In [ ]:
config = IngestorProviderConfig(name='reorder_columns', type='column', args={
    'column_order': ['status', 'role', 'username', 'user_id']
})
ingestor = ColumnReorderProvider(config)
result = ingestor.ingest(df)
print("ColumnReorderProvider:")
print(f"New column order: {list(result.columns)}")
print(result.head(2).to_string())

## Column Type Converter

Convert column data types before transformation. Ensures numeric fields are integers (not strings) and date fields are proper datetime objects for temporal queries.


In [ ]:
config = IngestorProviderConfig(name='convert_types', type='column', args={
    'type_mapping': {
        'access_level': 'int',
        'last_login': 'datetime'
    }
})
ingestor = ColumnTypeConverterProvider(config)
result = ingestor.ingest(df)
print("ColumnTypeConverterProvider:")
print(result.dtypes)
print(result[['access_level', 'last_login']].to_string())

## Row Filter (SkipRowProvider)

Filter rows by field conditions — keep only records that match your criteria. Useful for excluding inactive accounts, test data, or irrelevant record types.


In [ ]:
config = IngestorProviderConfig(name='filter_active', type='row', args={
    'conditions': [
        {'field': 'status', 'operator': 'eq', 'value': 'active'}
    ],
    'keep_matching': True
})
ingestor = SkipRowProvider(config)
result = ingestor.ingest(df)
print("SkipRowProvider (keep active only):")
print(result[['username', 'status']].to_string())
print(f"\nRows: {len(df)} -> {len(result)}")

## Date Range Filter

Filter records by a date field — keep only rows within a specified time window. Useful for incremental loads or time-bounded analysis.


In [ ]:
config = IngestorProviderConfig(name='recent_logins', type='row', args={
    'date_field': 'last_login',
    'start_date': '2026-06-01',
    'include_null': False
})
ingestor = DateRangeFilterProvider(config)
result = ingestor.ingest(df)
print("DateRangeFilterProvider (logins since 2026-06-01):")
print(result[['username', 'last_login']].to_string())
print(f"\nRows: {len(df)} -> {len(result)}")

## Numeric ID Cleanup

Remove the \ suffix that pandas adds when reading integer IDs from CSV (\ → \). Ensures clean, consistent node identifiers.


In [ ]:
config = IngestorProviderConfig(name='clean_ids', type='field', args={
    'field': 'user_id'
})
ingestor = NumericIdCleanupIngestor(config)
result = ingestor.ingest(df.copy())
print("NumericIdCleanupIngestor:")
print(f"Before: {df['user_id'].tolist()}")
print(f"After:  {result['user_id'].tolist()}")
print(f"Type:   {result['user_id'].dtype}")

## Summary
All ingestor providers tested successfully:
- **ColumnSelectorProvider**: Select/drop columns
- **ColumnRenamerProvider**: Rename columns via mapping
- **ColumnReorderProvider**: Reorder columns
- **ColumnTypeConverterProvider**: Convert column dtypes
- **SkipRowProvider**: Filter rows by conditions
- **DateRangeFilterProvider**: Filter by date range
- **NumericIdCleanupIngestor**: Remove .0 suffix from IDs